# Dataset Exploration and Preparation

Reproduces frozen dataset statistics from authoritative Step 1 evidence. It does not rescan the 3 GB raw CSV by default.

**Safety:** default execution is lightweight. Expensive full-data work runs only behind an explicit environment flag.

In [ ]:
from pathlib import Path
import os, json
import pandas as pd

candidate = Path(os.environ.get('ASSISTMENTS_WORKSPACE', Path.cwd().parents[1] / 'ASSISTments Dataset')).resolve()
if not candidate.exists():
    raise FileNotFoundError('Set ASSISTMENTS_WORKSPACE to the local ASSISTments Dataset folder')
WORKSPACE = candidate
print('Using workspace:', WORKSPACE)

In [ ]:
step1 = WORKSPACE / '01_ASSISTments_Dataset_Preparation' / 'outputs'
profile = json.loads((step1 / 'ASSISTments_Step1_Full_Profile.json').read_text(encoding='utf-8'))
dictionary = pd.read_csv(step1 / 'ASSISTments_Step1_Data_Dictionary.csv')
missing = pd.read_csv(step1 / 'ASSISTments_Step1_Missingness.csv')
print('Rows:', profile['dimensions']['rows'])
print('Original columns:', profile['dimensions']['source_columns'])
print('Learners:', profile['identifier_summary']['unique_learners'])
display(dictionary[['column_name','cleaned_dtype','documented_meaning','missing_count','missing_percentage']].head(12))

In [ ]:
assert profile['dimensions']['rows'] == 6_123_270
assert profile['dimensions']['source_columns'] == 35
assert profile['identifier_summary']['unique_learners'] == 46_674
correctness = pd.read_csv(step1 / 'distribution_correct.csv')
display(correctness)
print('Frozen evidence checks: PASS')

In [ ]:
RUN_RAW_SCAN = os.environ.get('RUN_RAW_SCAN') == '1'
raw = WORKSPACE / '2012-2013-data-with-predictions-4-final.csv'
if RUN_RAW_SCAN:
    print('WARNING: reading the raw file can be slow; this cell reads only 10,000 rows for schema inspection.')
    display(pd.read_csv(raw, nrows=10_000).head())
else:
    print('Raw scan skipped. Set RUN_RAW_SCAN=1 only when the raw CSV is present and a schema sample is needed.')